# Mastering AdaBoost: From Scratch to Scikit-Learn

Welcome to this comprehensive guide on **AdaBoost** (Adaptive Boosting). This notebook is designed for first-year B.Tech students and working professionals preparing for machine learning interviews.

### What this Notebook Covers:
- Theoretical foundation of AdaBoost.
- Complete implementation from scratch using purely NumPy.
- Scikit-Learn implementation and comparison.
- Hyperparameter tuning and visualization.
- Common interview questions and key takeaways.

### Prerequisites:
- Basic understanding of Python and NumPy.
- Familiarity with Decision Trees (specifically, Decision Stumps).
- Basic knowledge of binary classification.

### Dataset:
We are using the real-world **Titanic Dataset**. 
* **Kaggle Link:** [Titanic - Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

### Credits:
Authored by your friendly MIT Educator and FAANG ML Engineer.

### Why and What: Imports
**WHY:** We need external libraries to avoid reinventing the wheel for basic operations like data manipulation, scaling, and plotting.
**WHAT:** 
- `pandas` for data manipulation.
- `numpy` for core mathematical operations.
- `sklearn` for preprocessing, baseline models, and evaluation metrics.
- `matplotlib` and `seaborn` for data visualization.

In [ ]:
# Data manipulation and analysis
import pandas as pd

# Fundamental package for numerical computations
import numpy as np

# Machine learning: data splitting, models, evaluation, scaling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

# Plotting libraries for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility (Crucial for consistent results)
np.random.seed(42)

# Set styling for plots
sns.set_theme(style="whitegrid")

## Part 1: Theory Recap

Before we write code, let's understand the core concepts of AdaBoost.

### Real-World Analogy
Imagine studying for an exam using a deck of flashcards. After your first practice round, you realize you know some concepts perfectly, but missed others. For the next round, you pull the missed cards to the front of the deck so you see them more often. You repeat this. AdaBoost does exactly this: it increases the "weight" of data points it gets wrong so the next simple model focuses entirely on the hardest problems.

### Core Mathematics

1. **Final Ensemble Classifier:**
$$ F(x) = \text{sign}\left(\sum_{t=1}^{T} \alpha_t \cdot h_t(x)\right) $$
* $F(x)$ : Final binary prediction (+1 or -1)
* $T$ : Total number of weak learners (rounds)
* $\alpha_t$ : The Amount of Say (weight) of the $t$-th learner
* $h_t(x)$ : The prediction of the $t$-th learner

2. **Amount of Say (Alpha):**
$$ \alpha_t = \frac{1}{2} \ln\left(\frac{1 - \epsilon_t}{\epsilon_t}\right) $$
* $\alpha_t$ : Weight of the learner
* $\epsilon_t$ : Weighted error rate of the learner. If error is 0, $\alpha$ is huge. If error is 0.5 (random), $\alpha$ is 0.

3. **Sample Weight Update:**
$$ w_i^{(t+1)} = w_i^{(t)} \cdot \exp\left(-\alpha_t \cdot y_i \cdot h_t(x_i)\right) $$
* $w_i^{(t+1)}$ : New weight for sample $i$
* $w_i^{(t)}$ : Old weight for sample $i$
* $y_i$ : True label (+1 or -1)
* $h_t(x_i)$ : Predicted label (+1 or -1)
*(Note: If prediction is correct, the exponent is negative and weight shrinks. If incorrect, exponent is positive and weight grows!)*


### Why and What: Loading the Dataset
**WHY:** We need a real-world dataset to see how AdaBoost performs on noisy, non-synthetic data.
**WHAT:** We will load the Titanic dataset directly from a public URL. 
- **Features:** Information about the passengers (e.g., `Pclass` (ticket class), `Sex`, `Age`, `Fare`).
- **Target Variable:** `Survived` (0 = No, 1 = Yes). Our goal is binary classification.

In [ ]:
# Load the dataset directly from a reliable public repository
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("--- Dataset Head ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

print("\n--- Dataset Description ---")
display(df.describe())

### Why and What: Preprocessing
**WHY:** Machine learning models require numerical input and cannot handle NaN (missing) values natively.
**WHAT:** 
- **Drop Irrelevant Columns:** Remove `PassengerId`, `Name`, `Ticket`, and `Cabin` as they require complex NLP or have too many missing values.
- **Handle Missing Values:** Fill missing `Age` and `Fare` with the median, and `Embarked` with the mode.
- **Encode Categoricals:** Map `Sex` to integers and One-Hot Encode the `Embarked` column.
- **Scale Features:** Use `StandardScaler` to bring all features to a zero mean and unit variance.

In [ ]:
# 1. Drop irrelevant columns
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

# 2. Handle missing values
# Fill Age and Fare with median values
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
# Fill Embarked with the most frequent value (mode)
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# 3. Encode categorical variables
# Map Sex to 0 and 1
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
# One-hot encode Embarked (dropping the first category to avoid multicollinearity)
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)

# Separate features (X) and target (y)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values

# 4. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Custom AdaBoost Implementation expects targets to be -1 and 1 (Standard formulation)
y_train_custom = np.where(y_train == 0, -1, 1)
y_test_custom = np.where(y_test == 0, -1, 1)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## Part 2: From Scratch Implementation

### Why and What: Implementing AdaBoost from Scratch
**WHY:** Building the algorithm from scratch demystifies the "black box" of machine learning. It ensures you understand exactly how the sample weights and alphas are calculated, which is a highly tested concept in ML interviews.
**WHAT:** We will build two classes using purely NumPy:
1. `DecisionStump`: The weak learner. It finds a single feature and a threshold to split the data.
2. `AdaBoostScratch`: The main loop that trains stumps, calculates their $\alpha$ (Amount of Say), and updates the sample weights.

In [ ]:
class DecisionStump:
    def __init__(self):
        # Polarity determines the direction of the inequality for the split
        self.polarity = 1
        self.feature_idx = None
        self.threshold = None
        # Alpha is the "amount of say" (weight) of this stump
        self.alpha = None

    def predict(self, X):
        n_samples = X.shape[0]
        X_column = X[:, self.feature_idx]
        # Default predictions are 1
        predictions = np.ones(n_samples)
        
        # Apply the threshold to separate classes
        if self.polarity == 1:
            predictions[X_column < self.threshold] = -1
        else:
            predictions[X_column >= self.threshold] = -1
            
        return predictions

class AdaBoostScratch:
    def __init__(self, n_clf=50):
        self.n_clf = n_clf
        self.clfs = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # 1. Initialize weights evenly to 1/N
        w = np.full(n_samples, (1 / n_samples))
        
        self.clfs = []
        
        for _ in range(self.n_clf):
            clf = DecisionStump()
            min_error = float('inf')
            
            # Find the best feature and threshold to split on
            for feature_i in range(n_features):
                X_column = X[:, feature_i]
                thresholds = np.unique(X_column)
                
                for threshold in thresholds:
                    p = 1
                    predictions = np.ones(n_samples)
                    predictions[X_column < threshold] = -1
                    
                    # Error = sum of weights of misclassified samples
                    error = sum(w[y != predictions])
                    
                    # Flip polarity if error is greater than 0.5 (worse than random guessing)
                    if error > 0.5:
                        error = 1 - error
                        p = -1
                    
                    # Keep track of the stump that minimizes weighted error
                    if error < min_error:
                        min_error = error
                        clf.polarity = p
                        clf.threshold = threshold
                        clf.feature_idx = feature_i
            
            # INTERVIEW NOTE: Calculating the "Amount of Say" (Alpha)
            # A lower error gives a higher alpha. EPS prevents division by zero.
            EPS = 1e-10
            clf.alpha = 0.5 * np.log((1.0 - min_error + EPS) / (min_error + EPS))
            
            # Get predictions from the best stump
            predictions = clf.predict(X)
            
            # INTERVIEW NOTE: Weight Update Step
            # If y == predictions, exponent is negative -> weight decreases.
            # If y != predictions, exponent is positive -> weight increases!
            w *= np.exp(-clf.alpha * y * predictions)
            
            # Normalize the weights so they sum to 1
            w /= np.sum(w)
            
            self.clfs.append(clf)

    def predict(self, X):
        # Get predictions from all stumps, weighted by their alpha
        clf_preds = [clf.alpha * clf.predict(X) for clf in self.clfs]
        
        # Sum the weighted predictions
        y_pred = np.sum(clf_preds, axis=0)
        
        # Return the sign of the sum (-1 or 1)
        return np.sign(y_pred)

### Why and What: Training and Evaluating Scratch Implementation
**WHY:** To prove that our mathematical implementation functions correctly and learns complex patterns from the Titanic dataset.
**WHAT:** We instantiate our `AdaBoostScratch` with 50 estimators, fit it on the training data, predict on the test data, and display the Accuracy and Classification Report.

In [ ]:
# Initialize and train our custom AdaBoost
custom_adaboost = AdaBoostScratch(n_clf=50)
custom_adaboost.fit(X_train, y_train_custom)

# Predict on the test set
y_pred_custom = custom_adaboost.predict(X_test)

# Convert predictions back to 0 and 1 for metric evaluation
y_pred_custom_binary = np.where(y_pred_custom == -1, 0, 1)

# Evaluate performance
scratch_accuracy = accuracy_score(y_test, y_pred_custom_binary)
print(f"Scratch AdaBoost Accuracy: {scratch_accuracy * 100:.2f}%")
print("\nClassification Report (Scratch):")
print(classification_report(y_test, y_pred_custom_binary))

## Part 3: Scikit-Learn Implementation

### Why and What: Using sklearn
**WHY:** While building from scratch is great for learning, in production, we rely on highly optimized, tested libraries like Scikit-Learn. It runs significantly faster because it's implemented in Cython/C under the hood.
**WHAT:** We will use `sklearn.ensemble.AdaBoostClassifier`. We will set `algorithm="SAMME"` to closely mirror our discrete AdaBoost scratch implementation, fit the model, predict, and compare the accuracy.

In [ ]:
# Initialize Scikit-Learn's AdaBoost Classifier
# We use algorithm="SAMME" to align with standard discrete AdaBoost
sklearn_adaboost = AdaBoostClassifier(n_estimators=50, random_state=42, algorithm="SAMME")

# Fit the model using the standard 0/1 labels
sklearn_adaboost.fit(X_train, y_train)

# Predict on the test set
y_pred_sklearn = sklearn_adaboost.predict(X_test)

# Evaluate performance
sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)
print(f"Scikit-Learn AdaBoost Accuracy: {sklearn_accuracy * 100:.2f}%")
print("\nClassification Report (Scikit-Learn):")
print(classification_report(y_test, y_pred_sklearn))

print("-" * 40)
print(f"Accuracy Difference: {abs(sklearn_accuracy - scratch_accuracy) * 100:.2f}%")

### Why and What: Visualizations
**WHY:** Metrics like accuracy only tell part of the story. Visualizations help us diagnose model behavior and understand data features.
**WHAT:** We will plot:
1. **ROC Curve (Receiver Operating Characteristic):** Evaluates the tradeoff between True Positive Rate and False Positive Rate.
2. **Feature Importances:** Identifies which features (like `Sex` or `Fare`) played the biggest role in AdaBoost's decision making.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: ROC Curve
# Getting prediction probabilities for the positive class
y_pred_proba = sklearn_adaboost.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('Receiver Operating Characteristic (ROC)', fontsize=14)
axes[0].legend(loc="lower right")

# Plot 2: Feature Importances
importances = sklearn_adaboost.feature_importances_
features = df_clean.drop('Survived', axis=1).columns
indices = np.argsort(importances) # Ascending for horizontal bar chart

axes[1].barh(range(len(indices)), importances[indices], color='teal', align='center', label='Feature Importance')
axes[1].set_yticks(range(len(indices)))
axes[1].set_yticklabels([features[i] for i in indices])
axes[1].set_xlabel('Relative Importance', fontsize=12)
axes[1].set_ylabel('Features', fontsize=12)
axes[1].set_title('Feature Importances in AdaBoost', fontsize=14)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

### Why and What: Tuning AdaBoost
**WHY:** Models rarely perform optimally with default parameters. Understanding how hyperparameters interact is vital for optimizing performance.
**WHAT:** We will experiment with two crucial hyperparameters using 5-fold cross-validation:
1. `n_estimators`: The maximum number of weak learners. Too few causes underfitting; too many increases training time and can eventually lead to overfitting.
2. `learning_rate`: Shrinks the contribution of each classifier. There is a strong trade-off: a lower learning rate requires more estimators to maintain the same performance.

In [ ]:
n_estimators_range = [10, 50, 100, 200, 300]
learning_rates = [0.1, 0.5, 1.0]

results_df = pd.DataFrame(columns=['n_estimators', 'learning_rate', 'cv_accuracy'])

rows = []
for lr in learning_rates:
    for n in n_estimators_range:
        clf = AdaBoostClassifier(n_estimators=n, learning_rate=lr, random_state=42, algorithm="SAMME")
        # 5-fold cross-validation
        scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')
        rows.append({'n_estimators': n, 'learning_rate': lr, 'cv_accuracy': scores.mean()})

results_df = pd.DataFrame(rows)

# Plotting the impact
plt.figure(figsize=(10, 6))
sns.lineplot(data=results_df, x='n_estimators', y='cv_accuracy', hue='learning_rate', marker='o', palette='Set1')
plt.title('Impact of n_estimators and learning_rate on CV Accuracy', fontsize=14)
plt.xlabel('Number of Estimators (n_estimators)', fontsize=12)
plt.ylabel('Mean 5-Fold CV Accuracy', fontsize=12)
# Explicitly adding legend
plt.legend(title='Learning Rate', loc='lower right')
plt.grid(True)
plt.show()

## Part 5: Interview Corner

**Q: How does AdaBoost differ from Random Forest? Which is more susceptible to outliers and why?**

**A:** 
* **Tree Depth:** Random Forest builds fully grown, deep, independent decision trees. AdaBoost typically uses "Decision Stumps" (trees with a depth of 1).
* **Sequential vs Parallel:** Random Forest builds trees in parallel (Bagging). AdaBoost builds them sequentially (Boosting), where each new tree tries to correct the errors of the previous ones.
* **Weighting:** In Random Forest, all trees have an equal vote. In AdaBoost, trees are weighted based on their accuracy (the `alpha` parameter).
* **Outliers (The most critical difference):** **AdaBoost is highly susceptible to outliers.** Because it explicitly increases the weight of misclassified points at every step, a stubborn outlier (which is consistently misclassified) will monopolize the weights. Subsequent trees will focus entirely on trying to fit that noisy outlier, leading to severe overfitting. Random Forest simply averages out predictions, making it highly robust to noise and outliers.

## Key Takeaways

1. **Adaptive Focus:** AdaBoost sequentially adapts by giving higher weights to previously misclassified samples, forcing the algorithm to learn complex boundaries.
2. **Weak to Strong:** It proves that a combination of weak learners (barely better than random guessing) can form an incredibly strong classifier.
3. **Amount of Say:** The `alpha` value dynamically scales the voting power of each stump based on its training error—lower error means higher voting weight.
4. **Outlier Sensitivity:** Due to its aggressive weight updates, noisy data or outliers can completely derail the algorithm, making data cleaning absolutely crucial.
5. **Hyperparameter Trade-off:** There is an inverse relationship between `learning_rate` and `n_estimators`. A lower learning rate shrinks the contribution of each tree, requiring more trees to achieve comparable accuracy.